# Prática - Mapeamento de Texturas + MVP

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfw.terminate()
    
glfw.make_context_current(window)


### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [3]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [4]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print("Carregando textura:", img_textura)
    print("texture_id:", texture_id)
    print("OpenGL:", glGetString(GL_VERSION))
    print("glIsTexture antes:", glIsTexture(int(texture_id)))

    glBindTexture(GL_TEXTURE_2D, int(texture_id))

    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)

    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    img = Image.open(img_textura).convert("RGB")
    img = img.transpose(Image.FLIP_TOP_BOTTOM)

    img_width, img_height = img.size
    image_data = np.ascontiguousarray(np.array(img, dtype=np.uint8))

    print("Imagem:", img_width, img_height, image_data.shape, image_data.dtype)

    glTexImage2D(
        GL_TEXTURE_2D,
        0,
        GL_RGB,
        img_width,
        img_height,
        0,
        GL_RGB,
        GL_UNSIGNED_BYTE,
        image_data.tobytes()
    )

    print("Erro OpenGL depois:", glGetError())



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def triangula_face(arr):
    if len(arr) == 3:
        return arr

    result = []
    for i in range(1, len(arr) - 1):
        result.extend([arr[0], arr[i], arr[i + 1]])
    return result
    
global texture_ids
texture_ids = glGenTextures(20)
print("TEXTURE IDS:", texture_ids)
print("OPENGL:", glGetString(GL_VERSION))

global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in triangula_face(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in triangula_face(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(texture_ids[numberTextures], texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

TEXTURE IDS: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
OPENGL: b'4.6.0 - Build 30.0.101.1404'


### Vamos carregar cada modelo e definir funções para desenhá-los

In [5]:
# carrega objetos (modelo e texturas)
verticeInicial_plano, quantosVertices_plano = load_obj_and_texture('plano1.obj', ['grass.jpg'])
verticeInicial_cercado, quantosVertices_cercado = load_obj_and_texture('cercado.obj', ['woodText.jpg'])
verticeInicial_arvore, quantosVertices_arvore = load_obj_and_texture('arvore.obj', ['textTree2.jpeg'])
verticeInicial_caixaCorreio, quantosVertices_caixaCorreio = load_obj_and_texture('mailbox.obj', ['mailbox.png'])
verticeInicial_balanco1, quantosVertices_balanco1 = load_obj_and_texture('balanco1.obj', ['Wood_BaseColor.png'])
verticeInicial_balanco2, quantosVertices_balanco2 = load_obj_and_texture('balanco2.obj', ['Wood_BaseColor.png'])
verticeInicial_casaAbelha, quantosVertices_casaAbelha = load_obj_and_texture('beeHouse.obj', ['beeHouseText.png'])
verticeInicial_folhas, quantosVertices_folhas = load_obj_and_texture('folhas.obj', ['folhasText.png'])
verticeInicial_catavento, quantosVertices_catavento = load_obj_and_texture('catavento.obj', ['cataventoText.png'])
verticeInicial_cataventoCabo, quantosVertices_cataventoCabo = load_obj_and_texture('cataventoCabo.obj', ['cataventoCabo.jpg'])
verticeInicial_fonte, quantosVertices_fonte = load_obj_and_texture('fonte.obj', ['fonteText.png']) 
verticeInicial_flores, quantosVertices_flores = load_obj_and_texture('flores.obj', ['floresText.png']) 
verticeInicial_gnomo, quantosVertices_gnomo = load_obj_and_texture('gnomo.obj', ['gnomoText.jpg'])
verticeInicial_gnomo2, quantosVertices_gnomo2 = load_obj_and_texture('gnomo2.obj', ['gnomo2Text.png'])
verticeInicial_baloes, quantosVertices_baloes = load_obj_and_texture('ballons1.obj', ['16_256.png'])
verticeInicial_casa, quantosVertices_casa = load_obj_and_texture('house1.obj', ['16_256.png'])
verticeInicial_teto, quantosVertices_teto = load_obj_and_texture('ceiling.obj', ['WoodPlanks_diffuse.jpg'])
verticeInicial_chao, quantosVertices_chao = load_obj_and_texture('floor.obj', ['WoodPlanks_diffuse.jpg'])

def uv_skybox(col, row, margem=0.002):
    """
    A imagem sky.png está organizada como cruz:

          [topo]
    [left][front][right][back]
          [baixo]

    Ela tem 4 colunas e 3 linhas.
    col = 0..3
    row = 0..2, contando de cima para baixo.
    """

    u0 = col / 4 + margem
    u1 = (col + 1) / 4 - margem

    # como sua função load_texture_from_file faz FLIP_TOP_BOTTOM,
    # invertemos o V aqui
    v0 = 1 - ((row + 1) / 3) + margem
    v1 = 1 - (row / 3) - margem

    return [
        [u0, v0],
        [u1, v0],
        [u1, v1],
        [u0, v1],
    ]


def adiciona_face_mundo_texturizada(v1, v2, v3, v4, uv):
    global vertices_list, textures_coord_list

    vertices_list.extend([
        v1, v2, v3,
        v1, v3, v4
    ])

    textures_coord_list.extend([
        uv[0], uv[1], uv[2],
        uv[0], uv[2], uv[3]
    ])


def cria_cubo_mundo_texturizado():
    global vertices_list, textures_coord_list

    verticeInicial = len(vertices_list)

    # tamanho do mundo
    x_min, x_max = -150, 300
    y_min, y_max = -20, 220
    z_min, z_max = -180, 180

    A = [x_min, y_min, z_min]
    B = [x_max, y_min, z_min]
    C = [x_max, y_max, z_min]
    D = [x_min, y_max, z_min]

    E = [x_min, y_min, z_max]
    F = [x_max, y_min, z_max]
    G = [x_max, y_max, z_max]
    H = [x_min, y_max, z_max]

    uv_topo     = uv_skybox(1, 0)
    uv_esquerda = uv_skybox(0, 1)
    uv_frente   = uv_skybox(1, 1)
    uv_direita  = uv_skybox(2, 1)
    uv_fundo    = uv_skybox(3, 1)
    uv_baixo    = uv_skybox(1, 2)

    # face do fundo
    adiciona_face_mundo_texturizada(A, B, C, D, uv_fundo)

    # face da frente
    adiciona_face_mundo_texturizada(F, E, H, G, uv_frente)

    # face esquerda
    adiciona_face_mundo_texturizada(E, A, D, H, uv_esquerda)

    # face direita
    adiciona_face_mundo_texturizada(B, F, G, C, uv_direita)

    # teto
    adiciona_face_mundo_texturizada(D, C, G, H, uv_topo)

    # chão de baixo
    adiciona_face_mundo_texturizada(E, F, B, A, uv_baixo)

    verticeFinal = len(vertices_list)

    return verticeInicial, verticeFinal - verticeInicial


verticeInicial_mundo, quantosVertices_mundo = cria_cubo_mundo_texturizado()

# carrega a textura do céu
texture_skybox = texture_ids[numberTextures]
load_texture_from_file(texture_skybox, "sky.png")
numberTextures += 1

glBindTexture(GL_TEXTURE_2D, int(texture_skybox))
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)

def desenha_mundo(textureId):
    mat_model = model(
        0, 0, 0, 1,
        0, 0, 0,
        1, 1, 1
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glEnable(GL_DEPTH_TEST)
    glDepthMask(GL_TRUE)

  
    glDisable(GL_BLEND)
    glDrawArrays(GL_TRIANGLES, verticeInicial_mundo, quantosVertices_mundo)
    glEnable(GL_BLEND)

def desenha_caixa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_plano, quantosVertices_plano) ## renderizando

def desenha_cercado(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_cercado, quantosVertices_cercado) ## renderizando

def desenha_arvore(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_arvore, quantosVertices_arvore) ## renderizando

def desenha_caixaCorreio(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_caixaCorreio, quantosVertices_caixaCorreio) ## renderizando

def desenha_balanco1(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_balanco1, quantosVertices_balanco1) ## renderizando

def desenha_balanco2(t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    angulo_swing = 10 * math.sin(2.5 * glfw.get_time())

    mat_model = model_balanco2_animado(
        t_x, t_y, t_z,
        s_x, s_y, s_z,
        angulo_swing
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_balanco2, quantosVertices_balanco2)

def desenha_casaAbelha(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_casaAbelha, quantosVertices_casaAbelha) ## renderizando

def desenha_folhas(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_folhas, quantosVertices_folhas) ## renderizando

def desenha_catavento(t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    # velocidade do giro
    angulo_giro = 300 * glfw.get_time()

    mat_model = model_catavento_girando(
        t_x, t_y, t_z,
        s_x, s_y, s_z,
        angulo_giro
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_catavento, quantosVertices_catavento)

def desenha_cataventoCabo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_cataventoCabo, quantosVertices_cataventoCabo) ## renderizando

def desenha_fonte(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fonte, quantosVertices_fonte) ## renderizando
    

def desenha_flores(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_flores, quantosVertices_flores) ## renderizando

def desenha_gnomo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_gnomo, quantosVertices_gnomo) ## renderizando

def desenha_gnomo2(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_gnomo2, quantosVertices_gnomo2) ## renderizando

def desenha_baloes(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_baloes, quantosVertices_baloes)

def desenha_casa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_casa, quantosVertices_casa)

def desenha_teto(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_teto, quantosVertices_teto)

def desenha_chao(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_chao, quantosVertices_chao)

def desenha_com_matriz(verticeInicial, quantosVertices, textureId, mat_model):
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial, quantosVertices)
    
def desenha_casa_up_voando():
    mat_casa_voando = model_up_voando(
        111, 13.5, -6,
        5, 5, 5
    )

    # balões
    desenha_com_matriz(
        verticeInicial_baloes,
        quantosVertices_baloes,
        texture_ids[14],
        mat_casa_voando
    )

    # corpo da casa
    desenha_com_matriz(
        verticeInicial_casa,
        quantosVertices_casa,
        texture_ids[15],
        mat_casa_voando
    )

    # teto
    desenha_com_matriz(
        verticeInicial_teto,
        quantosVertices_teto,
        texture_ids[16],
        mat_casa_voando
    )

    # chão
    desenha_com_matriz(
        verticeInicial_chao,
        quantosVertices_chao,
        texture_ids[17],
        mat_casa_voando
    )

    

Processando modelo plano1.obj. Vertice inicial: 0
Processando modelo plano1.obj. Vertice final: 6
Carregando textura: grass.jpg
texture_id: 1
OpenGL: b'4.6.0 - Build 30.0.101.1404'
glIsTexture antes: 0
Imagem: 1359 1359 (1359, 1359, 3) uint8
Erro OpenGL depois: 0
Processando modelo cercado.obj. Vertice inicial: 6
Processando modelo cercado.obj. Vertice final: 8826
Carregando textura: woodText.jpg
texture_id: 2
OpenGL: b'4.6.0 - Build 30.0.101.1404'
glIsTexture antes: 0
Imagem: 3840 5760 (5760, 3840, 3) uint8
Erro OpenGL depois: 0
Processando modelo arvore.obj. Vertice inicial: 8826
Processando modelo arvore.obj. Vertice final: 12240
Carregando textura: textTree2.jpeg
texture_id: 3
OpenGL: b'4.6.0 - Build 30.0.101.1404'
glIsTexture antes: 0
Imagem: 1024 1024 (1024, 1024, 3) uint8
Erro OpenGL depois: 0
Processando modelo mailbox.obj. Vertice inicial: 12240
Processando modelo mailbox.obj. Vertice final: 13254
Carregando textura: mailbox.png
texture_id: 4
OpenGL: b'4.6.0 - Build 30.0.101.1

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [6]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [7]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [8]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [9]:
#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);


# camera
cameraPos = glm.vec3(111.26, 25.0, 45.0)
cameraFront = glm.normalize(glm.vec3(0.0, -4.0, -15.0))
cameraUp = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw = -90.0
pitch = -15.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode, voo_ativo, tempo_inicio_voo, casa_no_ar

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if key == glfw.KEY_F and action == glfw.PRESS:
        casa_no_ar = not casa_no_ar

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.02 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [10]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def model_balanco2_animado(t_x, t_y, t_z, s_x, s_y, s_z, angulo_swing):
    # rotação 270 no eixo Y para deixar o objeto virado certo
    base = glm.mat4(1.0)

    base = glm.translate(base, glm.vec3(t_x, t_y, t_z))
    base = glm.rotate(base, glm.radians(270), glm.vec3(0, 1, 0))
    base = glm.scale(base, glm.vec3(s_x, s_y, s_z))

    # pega os vértices do balanco2 para encontrar o topo dele
    verts = np.array(
        vertices_list[verticeInicial_balanco2 : verticeInicial_balanco2 + quantosVertices_balanco2],
        dtype=np.float32
    )

    min_v = verts.min(axis=0)
    max_v = verts.max(axis=0)

    # pivô local: topo do objeto, onde ficam presas as correntes
    pivo_local = glm.vec3(
        (min_v[0] + max_v[0]) / 2,
        max_v[1],
        (min_v[2] + max_v[2]) / 2
    )

    # converte o pivô local para coordenada do mundo
    pivo_world = glm.vec3(base * glm.vec4(pivo_local, 1.0))

    # eixo do balanço
    rot_base = glm.mat4(1.0)
    rot_base = glm.rotate(rot_base, glm.radians(270), glm.vec3(0, 1, 0))

    eixo_world = glm.normalize(glm.vec3(rot_base * glm.vec4(1, 0, 0, 0)))

    # matriz da animação ao redor do pivô
    anim = glm.mat4(1.0)
    anim = glm.translate(anim, pivo_world)
    anim = glm.rotate(anim, glm.radians(angulo_swing), eixo_world)
    anim = glm.translate(anim, glm.vec3(-pivo_world.x, -pivo_world.y, -pivo_world.z))

    matriz_final = anim * base

    return np.array(matriz_final)

def model_catavento_girando(t_x, t_y, t_z, s_x, s_y, s_z, angulo_giro):
    # matriz base: posiciona e escala o catavento
    base = glm.mat4(1.0)
    base = glm.translate(base, glm.vec3(t_x, t_y, t_z))
    base = glm.scale(base, glm.vec3(s_x, s_y, s_z))

    # pega os vértices do catavento para achar o centro da hélice
    verts = np.array(
        vertices_list[verticeInicial_catavento : verticeInicial_catavento + quantosVertices_catavento],
        dtype=np.float32
    )

    min_v = verts.min(axis=0)
    max_v = verts.max(axis=0)

    # pivô local no centro do objeto
    pivo_local = glm.vec3(
        (min_v[0] + max_v[0]) / 2,
        (min_v[1] + max_v[1]) / 2,
        (min_v[2] + max_v[2]) / 2
    )

    # transforma o pivô para coordenada de mundo
    pivo_world = glm.vec3(base * glm.vec4(pivo_local, 1.0))

    # eixo de giro da hélice
    # se girar errado, troque para (1,0,0) ou (0,1,0)
    eixo_world = glm.vec3(0, 0, 1)

    # gira em torno do centro da hélice
    anim = glm.mat4(1.0)
    anim = glm.translate(anim, pivo_world)
    anim = glm.rotate(anim, glm.radians(angulo_giro), eixo_world)
    anim = glm.translate(anim, glm.vec3(-pivo_world.x, -pivo_world.y, -pivo_world.z))

    matriz_final = anim * base

    return np.array(matriz_final)

def model_up_voando(t_x, t_y, t_z, s_x, s_y, s_z):
    global casa_no_ar, altura_voo_casa, ultimo_tempo_casa

    agora = glfw.get_time()
    dt = agora - ultimo_tempo_casa
    ultimo_tempo_casa = agora

    altura_maxima = 12.0
    velocidade = 3.0

    # se apertou F e casa_no_ar virou True, ela sobe
    if casa_no_ar:
        altura_voo_casa += velocidade * dt
        if altura_voo_casa > altura_maxima:
            altura_voo_casa = altura_maxima

    # se apertou F de novo e casa_no_ar virou False, ela desce
    else:
        altura_voo_casa -= velocidade * dt
        if altura_voo_casa < 0.0:
            altura_voo_casa = 0.0

    # fator de animação: 0 no chão, 1 no alto
    fator = altura_voo_casa / altura_maxima

    # se estiver no chão, fica exatamente parada como antes
    if altura_voo_casa == 0.0:
        M = glm.mat4(1.0)
        M = glm.translate(M, glm.vec3(t_x, t_y, t_z))
        M = glm.rotate(M, glm.radians(270), glm.vec3(0, 1, 0))
        M = glm.scale(M, glm.vec3(s_x, s_y, s_z))
        return np.array(M)

    # balanço leve enquanto está no ar
    balanco_y = fator * 0.20 * math.sin(2.0 * agora)
    balanco_x = fator * 0.12 * math.sin(1.2 * agora)
    balanco_z = fator * 0.12 * math.cos(1.0 * agora)

    inclinacao_z = fator * 1.5 * math.sin(1.2 * agora)
    inclinacao_x = fator * 1.0 * math.cos(1.0 * agora)

    M = glm.mat4(1.0)

    # translação principal: sobe/desce no eixo Y
    M = glm.translate(
        M,
        glm.vec3(
            t_x + balanco_x,
            t_y + altura_voo_casa + balanco_y,
            t_z + balanco_z
        )
    )

    # rotação original da casa
    M = glm.rotate(M, glm.radians(270), glm.vec3(0, 1, 0))

    # rotações pequenas de flutuação
    M = glm.rotate(M, glm.radians(inclinacao_z), glm.vec3(0, 0, 1))
    M = glm.rotate(M, glm.radians(inclinacao_x), glm.vec3(1, 0, 0))

    # escala original
    M = glm.scale(M, glm.vec3(s_x, s_y, s_z))

    return np.array(M)

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(
        cameraPos,
        cameraPos + cameraFront,
        cameraUp
    )
    return np.array(mat_view)

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 500.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Nesse momento, nós exibimos a janela!


In [11]:
glfw.show_window(window)

### Loop principal da janela.

In [12]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

casa_no_ar = False
altura_voo_casa = 0.0
ultimo_tempo_casa = glfw.get_time()

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()

    glClearColor(1.0, 1.0, 1.0, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    # desenha o cubo do céu primeiro
    desenha_mundo(texture_skybox)


    desenha_caixa(0.0, 0, 0, 1, 0, 0, -20, 1.5, 1.5, 1.5, texture_ids[0])
    desenha_cercado(0.0, 0, 0, 1, 9.65, 25, 15.5, 1.5, 1.5, 1.5, texture_ids[1])
    #desenha_arvore(0.0, 0, 0, 1, 80, 8,-2, 1.5, 1.5, 1.5, texture_ids[2])
    desenha_arvore(0.0, 0, 0, 1, 110, 8,-2, 1.5, 1.5, 1.5, texture_ids[2])
    desenha_caixaCorreio(180, 0, 1, 0, 115.5, 9.85,15.5, 0.13, 0.23, 0.13, texture_ids[3])
    desenha_balanco1(
        270, 0, 1, 0,
        127.5, 9.85, 2,
        0.025, 0.040, 0.025,
        texture_ids[4]
    )
    
    desenha_balanco2(
        127.5, 10, 2,
        0.025, 0.015, 0.025,
        texture_ids[4]
    )

    desenha_casaAbelha(90.0, 0, 1, 0, 95.4, 7.6,-1, 2.3, 2.3, 2.3, texture_ids[6])

    desenha_folhas(0.0, 0, 0, 1, 94.4, 7.6,1.5, 2.0, 2.0, 2.0, texture_ids[7])
    desenha_folhas(0.0, 0, 0, 1, 116.6, 7.6,15.5, 1.5, 1.5, 1.5, texture_ids[7])

    desenha_catavento(
        102, 7.6, 0.08,
        5.5, 5.5, 5.5,
        texture_ids[8]
    )
    
    desenha_cataventoCabo(
        0.0, 0, 0, 1,
        102, 7.6, 0.08,
        5.5, 5.5, 5.5,
        texture_ids[9]
    )

    desenha_fonte(
        0.0, 0, 0, 1,
        122, 7.6, 8.5,
        0.8, 0.8, 0.8,
        texture_ids[10]
    )

    desenha_flores(
        0, 0, 0, 1,
        95, 7.8, 10.5,
        0.3, 0.3, 0.3,
        texture_ids[11]
    )

    desenha_gnomo(
        0.0, 0, 0, 1,
        120.6, 10.6, 8.9,
        5, 5, 5,
        texture_ids[12]
    )

    desenha_gnomo2(
        0.0, 0, 0, 1,
        119.6, 10.6, 9.2,
        0.4, 0.4, 0.4,
        texture_ids[13]
    )

    """
    desenha_baloes(
        270, 0, 1, 0,
        111, 13.5, -6,
        5, 5, 5,
        texture_ids[14]
    )
    desenha_casa(
        270, 0, 1, 0,
        111, 13.5, -6,
        5, 5, 5,
        texture_ids[15]
    )
    desenha_teto(
        270, 0, 1, 0,
        111, 13.5, -6,
        5, 5, 5,
        texture_ids[16]
    )
    desenha_chao(
        270, 0, 1, 0,
        111, 13.5, -6,
        5, 5, 5,
        texture_ids[17]
    )

    """

    desenha_casa_up_voando()

    

   

    glfw.swap_buffers(window)

glfw.terminate()